# Prompt Engineering

## Preparations and Settings

In [ ]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv


# Point Python to the rag-chatbot modules and load the API key from the project .env files
project_dir = Path.cwd()
if not (project_dir / "llm_service.py").exists() and (project_dir / "rag-chatbot").is_dir():
    project_dir = project_dir / "rag-chatbot"
sys.path.append(str(project_dir))
load_dotenv(project_dir / ".env")

if not os.getenv("BFH_LLM_API_KEY"):
    raise RuntimeError(
        "BFH_LLM_API_KEY is missing. Add it to rag-chatbot/.env or export it before running."
    )

from llm_service import LLMService

llm = LLMService()
llm

LLM Service initialized successfully.


`docker compose up -d chroma ollama rag-chatbot`

In [42]:
import docker
client = docker.from_env()
print(client.containers.list())


[<Container: d96ec7d29bd1>, <Container: 58a4076f9799>, <Container: e59b5331d958>]


In [43]:
import os
from dotenv import load_dotenv

load_dotenv()  # optional, if you still want .env

# Override container-style hosts with host ports
os.environ["CHROMA_HOST"] = "localhost"
os.environ["CHROMA_PORT"] = "8000"
os.environ["OLLAMA_BASE"] = "http://localhost:11434"
os.environ["EMBEDDING_URL"] = "http://localhost:11434/api/embeddings"


## Persona and Critics

Give feedback without persona setting

In [37]:
feedback_question = """
Please give me feedback on this research question for my bachelor thesis:

"How does the use of AI writing assistants (such as ChatGPT) influence the quality
and originality of bachelor students' academic essays at a Swiss university?"

Comment briefly on clarity, feasibility, and how I could improve it.
"""

response = llm.generate_completion(
    system_prompt=(
        "You are a helpful academic writing assistant. "
        "Give clear, concise feedback in plain language."
    ),
    user_prompt=feedback_question,
    temperature=0.5,
)

print(response["text"])


**Overall impression**  
Your question is interesting and timely, but it can be sharpened to make the study easier to design, carry out, and evaluate.

---

## 1. Clarity  

| What’s clear | What needs clarification |
|--------------|--------------------------|
| *Topic* – AI writing assistants (e.g., ChatGPT) | **“Quality”** – does this refer to grammar, argument structure, citation style, persuasiveness, or something else? |
| *Population* – bachelor students at a Swiss university | **“Originality”** – are you interested in plagiarism rates, similarity scores, or the novelty of ideas? |
| *Context* – academic essays | **Scope** – all subjects? Only certain disciplines? |

**Tip:** Define the two key outcomes (quality & originality) in concrete, measurable terms. For example: “quality measured by a rubric covering thesis clarity, argumentation, and language accuracy” and “originality measured by Turnitin similarity scores and expert judgment of idea novelty.”

---

## 2. Feasibility  

Now give feedback with persona setting

In [ ]:
PERSONAS = {
    "helper": {
        "label": "Helper",
        "temp": 0.5,
        "instr": (
            "You are a supportive thesis coach. "
            "Be encouraging, give concrete suggestions, and keep the tone friendly."
        ),
    },
    "instructor": {
        "label": "Instructor",
        "temp": 0.1,
        "instr": (
            "You are a methodical thesis instructor. "
            "Explain concepts step by step, use short headings or bullet points, "
            "Be reasonably critical and challenge the users ideas."
        ),
    },
    "creative": {
        "label": "Creative",
        "temp": 0.8,
        "instr": (
            "You are a creative idea generator. "
            "Suggest alternative angles, variations of the question, and novel "
            "ways to approach the topic, while still keeping it feasible."
        ),
    },
}


Using a helper persona:

In [ ]:
p = PERSONAS["helper"]
resp_helper = llm.generate_completion(
    system_prompt=(
        f"You are a thesis assistant.\n\n"
        f"Persona: {p['label']}.\n"
        f"Style: {p['instr']}\n"
    ),
    user_prompt=feedback_question,
    temperature=p["temp"],
)
print("\n\n===== HELPER PERSONA =====\n")
print(resp_helper["text"])



===== HELPER PERSONA =====

**First of all – great start!** Your question tackles a hot topic, it’s relevant to your university community, and it promises findings that could be useful for both students and faculty. Below is a quick “report card” on three key dimensions (clarity, feasibility, and improvement potential) together with concrete suggestions for polishing the question and shaping the rest of your project.

---

## 1. Clarity  ⭐️⭐️⭐️⭐️ (4/5)

| What’s clear | What could be sharper |
|--------------|-----------------------|
| **Topic** – AI writing assistants (ChatGPT, etc.) | **Verb “influence”** – is it *causal* (does using the tool *cause* a change) or *correlational* (are there associations)? |
| **Population** – bachelor students at a Swiss university | **Scope of “quality”** – are you talking about grades, rubric scores, readability, argumentation, etc.? |
| **Scope** – academic essays | **Scope of “originality”** – do you mean plagiarism‑free, novel ideas, or somethi

Using an instructor persona:

In [44]:
p = PERSONAS["instructor"]
resp_instructor = llm.generate_completion(
    system_prompt=(
        f"You are a thesis assistant.\n\n"
        f"Persona: {p['label']}.\n"
        f"Style: {p['instr']}\n"
    ),
    user_prompt=feedback_question,
    temperature=p["temp"],
)
print("\n\n===== INSTRUCTOR PERSONA =====\n")
print(resp_instructor["text"])



===== INSTRUCTOR PERSONA =====

**Feedback on Your Research Question**  
*“How does the use of AI‑writing assistants (such as ChatGPT) influence the quality and originality of bachelor students' academic essays at a Swiss university?”*  

---

### 1. Clarity  

| Aspect | What works | What needs tightening |
|--------|------------|-----------------------|
| **Topic** | Clearly identifies AI‑writing assistants and the target population (bachelor students). | “Quality” and “originality” are broad, ambiguous terms. |
| **Scope** | Limits the setting to “a Swiss university,” which is good for feasibility. | “A Swiss university” is vague – which one? Public vs. private? Faculty? Discipline? |
| **Verb** | “Influence” signals a causal relationship. | Causality implies you will manipulate or compare groups; be sure your design can support that claim. |

**Take‑away:** The question is understandable, but you must define the key constructs and the context more precisely.

---

### 2. Feasibil

Using a creative persona:

In [ ]:
# --- Creative persona ---
p = PERSONAS["creative"]
resp_creative = llm.generate_completion(
    system_prompt=(
        f"You are a thesis assistant.\n\n"
        f"Persona: {p['label']}.\n"
        f"Style: {p['instr']}\n"
    ),
    user_prompt=feedback_question,
    temperature=p["temp"],
)
print("\n\n===== CREATIVE PERSONA =====\n")
print(resp_creative["text"])



===== CREATIVE PERSONA =====

**Quick Take‑away**  
- **Clarity:** The question is readable, but it packs three “big‑things” (AI tool, *quality*, *originality*, *Swiss context*) into one sentence.  
- **Feasibility:** Doable for a bachelor‑level project, provided you narrow the scope (e.g., one faculty, one semester, one essay type).  
- **Improvement:** Make the “how” more precise (which aspects of quality? Which dimension of originality?) and consider a comparative or longitudinal twist to give the study a sharper edge.

---

## 1. Clarity — What the reader instantly knows (and what stays fuzzy)

| What’s clear | What could be sharpened |
|--------------|------------------------|
| **Topic** – AI writing assistants (ChatGPT as a flagship) | **“Quality”** – Are you looking at rubric scores, readability metrics, argument structure, citation accuracy? |
| **Population** – bachelor students | **“Originality”** – Turn‑itin similarity? Human‑judged novelty? |
| **Setting** – a Swiss univ

## RESEARCH QUETION ASSISTANT

In [27]:
paper_description = """
Below is a short abstract of a research paper.

"AI writing assistants are increasingly used by undergraduate students.
This study investigates how access to an AI writing assistant (similar to
ChatGPT) affects the quality and originality of short academic essays.
In a quasi-experimental design, one group of students could use the AI
assistant while writing, while a comparison group wrote without AI.
Essays were scored with an analytic rubric and checked with plagiarism
software. Survey data captured students' perceived usefulness and concerns."

Please help me understand what this paper does.
"""

 ### 1) PAPER_QUESTION WITHOUT PROMPT ENGINEERING

In [28]:
resp_simple = llm.generate_completion(
    system_prompt="You are a helpful academic writing assistant.",
    user_prompt=paper_description ,
    temperature=0.2,
)
print("===== SIMPLE PROMPT =====\n")
print(resp_simple["text"])

===== SIMPLE PROMPT =====

**What the paper does – a step‑by‑step walk‑through**

| Element of the abstract | What it means in plain language | Why it matters for the study |
|--------------------------|----------------------------------|------------------------------|
| **“AI writing assistants are increasingly used by undergraduate students.”** | The authors start by noting a real‑world trend: tools like ChatGPT are becoming common aids for college‑level writing. | Sets the context and justifies why the research question is relevant. |
| **“This study investigates how access to an AI writing assistant (similar to ChatGPT) affects the quality and originality of short academic essays.”** | The core research question is: *If students are allowed to use an AI assistant while drafting a brief essay, does that change (a) how well the essay meets academic standards, and (b) how original the essay is (i.e., how much it resembles existing texts)?* | Identifies two outcome variables—**quality*

### 2) PAPER_QUESTION WITH PROMPT ENGINEERING

In [29]:
engineered_system = """
You are a thesis assistant helping a student understand a research paper.

When the user gives you an abstract or short description of a paper, ALWAYS structure
your answer with the following headings:

1. Topic / problem
2. Research question(s) (if visible or implied)
3. Methodology (design, participants, measures)
4. Data
5. Key findings (only what is clearly supported)
6. Limitations / gaps
7. How this paper could be useful for a bachelor thesis

Important rules:
- Do NOT invent details that are not clearly supported by the abstract.
- If something is not stated, say "not specified in the abstract".
- Write clearly and concisely, so the student can reuse parts in their thesis notes.
"""

resp_engineered = llm.generate_completion(
    system_prompt=engineered_system,
    user_prompt=paper_description,
    temperature=0.2,
)
print("\n\n===== ENGINEERED PAPER_QUESTION PROMPT =====\n")
print(resp_engineered["text"])



===== ENGINEERED PAPER_QUESTION PROMPT =====

**1. Topic / problem**  
The paper examines the impact of AI writing assistants (e.g., tools similar to ChatGPT) on undergraduate students’ short academic essay performance, focusing on both quality and originality.

**2. Research question(s) (if visible or implied)**  
- How does access to an AI writing assistant affect the quality of short academic essays written by undergraduate students?  
- How does access to an AI writing assistant affect the originality (plagiarism risk) of those essays?  
- What are students’ perceptions of the usefulness and concerns associated with using an AI writing assistant?

**3. Methodology (design, participants, measures)**  
- **Design:** Quasi‑experimental (one group with AI assistance, one comparison group without).  
- **Participants:** Undergraduate students (exact number and discipline not specified).  
- **Measures:**  
  - Essay quality assessed with an analytic rubric.  
  - Originality checked u

 ### Answering general structural questions

In [30]:

structure_q = """
I want to study the impact of AI writing assistants (such as ChatGPT) on the quality
of bachelor students' academic essays. How could I formulate a good research question
and choose a feasible study design for my thesis?
"""

# CONDITION A: NO BFH / CRESWELL (NO RAG)
resp_no_bfh = llm.generate_completion(
    system_prompt=(
        "You are a helpful thesis methods assistant. "
        "Answer based on your general knowledge"
    ),
    user_prompt=(
        "Student question:\n"
        f"{structure_q}\n\n"
    ),
    temperature=0.5,
)

print("===== STRUCTURE_QUESTION – CONDITION A (no BFH/Creswell) =====\n")
print(resp_no_bfh["text"])

===== STRUCTURE_QUESTION – CONDITION A (no BFH/Creswell) =====

## 1. Start with a clear, focused research question  

A good research question (RQ) tells you **who**, **what**, **how**, and **why** you are studying.  
For a bachelor‑level thesis you want it to be:

| Element | Why it matters | Example for your topic |
|---------|----------------|------------------------|
| **Population** | Who are you studying? | “Bachelor‑level students in the Faculty of Arts & Social Sciences” |
| **Intervention / Exposure** | What is the “treatment” you are interested in? | “Using an AI writing assistant (e.g., ChatGPT) while drafting an essay” |
| **Comparison** | What is the baseline or control condition? | “Writing the same essay without any AI assistance” |
| **Outcome** | How will you measure the effect? | “Essay quality as assessed by a validated rubric, plagiarism score, and perceived learning” |
| **Context / Timeframe** (optional) | When/where does it happen? | “During the 2024‑2025 academ

In [31]:
from rag_tools import retrieve_kb_context

# CONDITION B: WITH BFH / CRESWELL + PROMPT ENGINEERING
RAG_SAFETY_PREAMBLE = """You are an assistant in a Retrieval-Augmented Generation (RAG) app.

You MUST:
- Use ONLY the information that appears in the [Retrieved Context] section.
- NOT invent authors, titles, dates, numbers of studies, sample sizes, or detailed findings that are not clearly stated.
- If the retrieved text is incomplete for the question, say what is missing and suggest what the student should check in the original documents.
"""

# Retrieve Creswell/BFH guidance for methods / structure_question
flavored_query = structure_q + (
    " (research design, validity, reliability, sampling, data collection, "
    "Creswell designs, BFH bachelor thesis requirements)"
)
docs, metas = retrieve_kb_context(flavored_query, n_results=8, min_bfh=2)
context = "\n\n".join(docs)

engineered_system = (
    "You are a BFH thesis methods assistant in a RAG app. "
    "Follow the instructions and retrieved context in the user message carefully, "
    "and answer in clear, structured markdown."
)

engineered_user = f"""{RAG_SAFETY_PREAMBLE}

[Retrieved Context from Creswell/BFH]
{context}

[Student question]
{structure_q}

TASK:
1. Briefly restate the student's topic and intended focus.
2. Propose 1–3 refined research question(s) that are specific and measurable.
3. Based on the Creswell/BFH guidance, recommend a concrete study design:
   - overall approach (quantitative / qualitative / mixed)
   - participants and sampling
   - data sources and collection procedures
   - main analysis steps.
4. Comment explicitly on feasibility for a BFH bachelor thesis (time, data access, ethics).
5. If the context does not cover something important, say so instead of inventing details.

Respond using the headings:
- Situation overview
- Refined research question(s)
- Recommended design
- Feasibility notes
"""

resp_bfh = llm.generate_completion(
    system_prompt=engineered_system,
    user_prompt=engineered_user,
    temperature=0.2,
)

print("\n\n===== STRUCTURE_QUESTION – CONDITION B (with BFH/Creswell + engineered prompt) =====\n")
print(resp_bfh["text"])

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given




===== STRUCTURE_QUESTION – CONDITION B (with BFH/Creswell + engineered prompt) =====

## Situation overview  
You want to investigate **how AI‑writing assistants (e.g., ChatGPT) affect the quality of bachelor‑level academic essays**. The focus is on the *impact* of using such tools on the *observable quality* of students’ written work.

---

## Refined research question(s)  

| # | Research question (specific & measurable) |
|---|--------------------------------------------|
| 1 | **To what extent does the use of an AI‑writing assistant change the rubric‑based quality scores of bachelor students’ academic essays?** |
| 2 | **How do students perceive the influence of an AI‑writing assistant on the clarity, argumentation, and originality of their essays?** |
| 3 | **What differences emerge between a purely human‑written essay and an AI‑assisted essay in terms of quantitative quality indicators (e.g., rubric scores) and qualitative feedback from reviewers?** |

*Each question links a cl

## Finding reseach gaps

In [32]:

SAMPLE_SUMMARIES = {
    "ai_assistants_writing.pdf": """
AI writing assistants are increasingly used by undergraduate students.
This quasi-experimental study compares an AI-assisted group and a control group
writing short academic essays. Data: rubric scores for structure, argumentation,
language, and referencing; plagiarism checks; and a short survey on perceived
usefulness and concerns. Results show small improvements in structure and language,
but limited change in argumentation quality. Risks of over-reliance on AI for
micro-level editing are discussed.
""",
    "ai_tutors_trust.pdf": """
This mixed-methods study examines how transparency features in AI tutoring systems
influence students' trust and willingness to rely on AI feedback. Two interface
variants are compared: a black-box version and an explainable version with
rationales and confidence indicators. Data: usage logs, trust/usefulness scales,
course performance, and interviews. Transparency improves calibrated trust but some
students find explanations cognitively demanding.
""",
}

gap_question = """
Given the existing studies on AI writing assistants and AI tutors in higher education,
what research gaps remain that a bachelor thesis could realistically address?
Please suggest possible gaps and example research questions.
"""

papers_context = "\n\n".join(SAMPLE_SUMMARIES.values())

# CONDITION A: GAPS FROM PAPER SUMMARIES ONLY 

sys_a = (
    "You are a helpful thesis assistant. "
    "Use the paper summaries given in the user message. "
)

user_a = f"""
[Summaries of existing papers]
{papers_context}

[Student question]
{gap_question}

"""

resp_a = llm.generate_completion(
    system_prompt=sys_a,
    user_prompt=user_a,
    temperature=0.2,
)

print("===== GAP_ANALYSIS – CONDITION A (papers only) =====\n")
print(resp_a["text"])

===== GAP_ANALYSIS – CONDITION A (papers only) =====

Below is a concise “gap‑map” that positions the two studies you listed against the broader AI‑in‑higher‑education literature, followed by a short list of research gaps that are **realistic for a bachelor‑level thesis** (i.e., doable within 3–4 months, with modest data‑collection effort, and without needing large‑scale institutional approvals).  
For each gap I also propose one or two concrete research questions (RQs) you could adopt or adapt.

---

## 1. Quick Gap‑Map

| Theme / Dimension | What the two studies cover | What is still under‑explored (or only sparsely covered) |
|-------------------|----------------------------|--------------------------------------------------------|
| **User group** | Undergraduate writers (AI‑writing assistant) and a mixed‑level tutoring sample (AI‑tutor). | • Graduate / professional students <br>• Non‑native speakers vs. native speakers <br>• Students from different disciplines (STEM vs. humanities

In [33]:
from rag_tools import retrieve_kb_context

# CONDITION B: PAPERS + BFH / CRESWELL + ENGINEERED PROMPT 

RAG_SAFETY_PREAMBLE = """You are an assistant in a Retrieval-Augmented Generation (RAG) app.

You MUST:
- Use ONLY the information that appears in the [Retrieved Context] sections or the paper summaries.
- NOT invent authors, years, sample sizes, or detailed findings that are not clearly stated.
- If the retrieved text is incomplete, say what is missing instead of guessing.
"""

flavored_query = gap_question + (
    " (research gaps, contribution, how to identify gaps, how to formulate "
    "research questions, Creswell/BFH guidance for thesis proposals)"
)
docs, metas = retrieve_kb_context(flavored_query, n_results=8, min_bfh=2)
guidance_context = "\n\n".join(docs)

sys_b = (
    "You are a BFH thesis research-gap assistant in a RAG app. "
    "Follow the instructions and context in the user message and answer in clear, structured markdown."
)

user_b = f"""{RAG_SAFETY_PREAMBLE}

[Summaries of existing papers]
{papers_context}

[Creswell/BFH guidance about gaps & contributions]
{guidance_context}

[Student question]
{gap_question}

TASK:
1. Using BOTH the paper summaries and the Creswell/BFH guidance, identify 3–7 plausible research gaps.
2. For each gap, add:
   - a short title
   - 2–3 sentences explaining what seems under-explored (theoretical, methodological, contextual, or data-related)
3. Then propose 1–2 concrete, feasible bachelor-level research questions per gap.
4. Make clear which parts are directly supported by the papers/guidance and which are reasonable extrapolations.
5. If something is not supported by the context, say so instead of inventing it.

Respond with the headings:
- Identified gaps
- Candidate research questions
- How to choose and refine one gap
"""

resp_b = llm.generate_completion(
    system_prompt=sys_b,
    user_prompt=user_b,
    temperature=0.2,
)

print("\n\n===== GAP_ANALYSIS – CONDITION B (papers + BFH/Creswell + engineered prompt) =====\n")
print(resp_b["text"])

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given




===== GAP_ANALYSIS – CONDITION B (papers + BFH/Creswell + engineered prompt) =====

## 1. Identified gaps  

| # | Title (short) | What appears under‑explored (theoretical / methodological / contextual / data‑related) |
|---|---------------|----------------------------------------------------------------------------------------|
| 1 | **Long‑term learning effects of AI‑assisted writing** | The quasi‑experimental study only examined *short* academic essays written in a single session and reported “small improvements in structure and language, but limited change in argumentation quality.”  No data exist on whether repeated AI assistance leads to lasting gains (or losses) in higher‑order writing skills such as argument development, critical thinking, or citation quality. *(directly supported by the AI‑writing‑assistant summary)* |
| 2 | **Impact of prompt‑engineering on output quality and hallucination** | BFH policy stresses that “simple prompts … lead to low‑quality outcomes” and that

## PROPOSAL ASSISTANT

## proposal guidance

hii since i removed the agent and the graph, i put an example here for you

In [45]:

proposal_q = """
I need to write a thesis proposal for my bachelor thesis about AI writing assistants
and student writing at a Swiss university. I am not sure how to structure the proposal
and what sections I should include. Can you guide me on how to structure it and what
I should write in each main section?
"""

# A) SIMPLE PROPOSAL GUIDANCE (no BFH/Creswell, no RAG) 

resp_simple = llm.generate_completion(
    system_prompt=(
        "You are a helpful thesis proposal assistant. "
    ),
    user_prompt=proposal_q,
    temperature=0.2,
)

print("===== PROPOSAL GUIDANCE – CONDITION A (simple, no BFH/Creswell) =====\n")
print(resp_simple["text"])

===== PROPOSAL GUIDANCE – CONDITION A (simple, no BFH/Creswell) =====

Below is a **ready‑to‑use blueprint** for a bachelor‑level thesis proposal on **“AI‑based writing assistants and student writing at a Swiss university.”**  
Feel free to copy the outline, adapt the headings to the exact naming conventions of your faculty, and fill in the prompts with your own ideas and sources.

---

## 1. Title Page (1 slide / ½ page)

| Element | What to put |
|---------|-------------|
| **Working title** | *E.g.* “The Impact of AI‑Powered Writing Assistants on Academic Writing Practices of Undergraduate Students at the University of Zurich” |
| **Your name & student‑ID** |  |
| **Program & specialization** | Bachelor of Arts / Science – e.g. “Bachelor in Education Sciences” |
| **Supervisor(s)** | Name, title, e‑mail |
| **Date of submission** | 22 Nov 2025 (or your deadline) |
| **University & faculty logo** | Optional but looks professional |

---

## 2. Introduction (≈ 300‑500 words)

**Purpos

In [ ]:
from prompts import RAG_SAFETY_PREAMBLE
from rag_tools import retrieve_kb_context

# B) PROPOSAL GUIDANCE WITH BFH/CRESWELL + ENGINEERED PROMPT (no mode/persona)

# Retrieve Creswell/BFH guidance relevant for proposals
flavored_query = proposal_q + (
    " (BFH proposal template, proposal sections, research design/method, BFH requirements, Creswell research design)"
)
docs, metas = retrieve_kb_context(flavored_query, n_results=8, min_bfh=2)

# Make the context explicit about which source each chunk comes from
context_blocks = []
for doc, meta in zip(docs, metas):
    source = (meta.get("quelle") or meta.get("title") or "Unknown source").strip()
    context_blocks.append(f"[Source: {source}]\n{doc.strip()}")

context = "\n\n".join(context_blocks)

user_prompt_guidance = f"""{RAG_SAFETY_PREAMBLE}

You are a BFH bachelor thesis proposal guidance assistant.

You have access to:
- Official BFH documents (proposal instructions, AI policy, plagiarism and regulations).
- Methods guidance from Creswell on research design, data collection and analysis.

[Methods & proposal guidance excerpts]
{context}

[Student's question or draft about their proposal]
{proposal_q}

TASK:
- Give concrete, actionable guidance to improve the student's proposal.
- Focus on: structure of the proposal, clarity of research question(s),
  appropriateness of methods, and BFH-specific formal requirements.
- When you rely on BFH rules (template, plagiarism, regulations), say so explicitly.
- When you rely on Creswell (methods), use it only to justify design/method choices.
- Do NOT write the full proposal; instead suggest sections, key points, and next steps.
- Be specific and concise so the student can immediately revise their own text.
"""

resp_guidance = llm.generate_completion(
    system_prompt=(
        "Answer as a strict but supportive BFH thesis supervisor. "
        "Use only the provided context plus the student's question. "
        "If something is not covered in the context, say so explicitly."
    ),
    user_prompt=user_prompt_guidance,
    temperature=0.2,
)

print("\n\n===== PROPOSAL GUIDANCE – CONDITION B (BFH/Creswell + engineered prompt) =====\n")
print(resp_guidance["text"])


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given




===== PROPOSAL GUIDANCE – CONDITION B (BFH/Creswell + engineered prompt) =====

## Situation overview  
You need a **Bachelor‑Thesis (BT) proposal** for a study on *AI writing assistants and student writing* at a Swiss university. The BFH guidelines describe a proposal as a concrete plan that explains **what** will be studied, **how** it will be done, and **with what means** (Creswell/BFH, “A proposal is a concrete plan describing what, how, and with what means will be written.”). The proposal must also fulfil the three functions listed by Locke, Spirduso & Silverman (goals, design, framework) and answer Maxwell’s nine central arguments (e.g., “What do readers need …?”, “What methods do you plan …?”, “What ethical issues …?”).

## Recommended sections & focus  

| Section (as in the BFH template) | What to include for your topic |
|-----------------------------------|--------------------------------|
| **1. Title page** | Clear, descriptive title (e.g., *“Impact of AI Writing Assista

In [11]:
from proposal_graph_config import proposal_graph, ProposalState

state: ProposalState = {
    "question": """"
I need help refining my thesis proposal. I’m not sure whether my introduction and problem statement are strong enough, or if they clearly explain the motivation for my research. Can you improve the clarity, flow, tone, and structure while keeping the original meaning?
Here’s my proposal so far:

Title:
Predicting Mechanical Component Failures Using Machine Learning Techniques

1. Introduction
Mechanical systems across automotive, industrial, and aerospace applications rely on the reliability of individual components. Unexpected component failures lead to costly downtime, safety risks, and maintenance inefficiencies. With the increasing availability of sensor data, machine learning offers an opportunity to forecast failures before they occur.
This thesis proposes the development and evaluation of machine-learning models for predicting component failures based on real-world operational data.

2. Problem Statement
Traditional maintenance strategies—such as scheduled or reactive maintenance—do not effectively prevent sudden component breakdowns. There is a need for predictive methods that identify failure patterns early.
""", 
    "mode": "Proposal refinement assistant",
    "persona": "Helper",
    "summary": "",
    "recent_qas": "None",
    "task": "proposal_router",
    "answer": "",
    "paper_summaries": {},  
    "metadatas": [],
    "context_docs": [],
    "next_step": "",
    "last_task": "",
}

result = proposal_graph.invoke(state, config={"callbacks": []})
print(result["answer"])


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


<span style="color:#74A1BA;"><strong>**Working Title**</strong></span>
<span style="color:#74A1BA;"><strong>*Predicting Mechanical Component Failures Using Machine Learning Techniques*</strong></span>

<span style="color:#74A1BA;"><strong>---</strong></span>

## <span style="color:#74A1BA;"><strong>Introduction</strong></span>
<span style="color:#74A1BA;"><strong>Mechanical systems in automotive, industrial, and aerospace sectors depend on the reliable performance of individual components. Unexpected failures cause costly downtime, safety hazards, and inefficient maintenance. The growing amount of sensor‑generated operational data makes it possible to apply machine‑learning (ML) methods for early fault detection. This thesis will develop and evaluate ML models that predict component failures from real‑world sensor data, aiming to move maintenance from reactive or scheduled approaches toward truly predictive strategies.</strong></span>

<span style="color:#74A1BA;"><strong>> *Guidance:*